# CDS change type alluvial (Ensembl → CAT)
Protein-coding only; grouped to 5 categories by default.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RESULTS_DIR = Path('../results')
DIV_DIR = RESULTS_DIR / 'intermediate_spreadsheets' / 'divergence'
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(exist_ok=True)

GROUPED = True  # toggle to False for 8-type view

GROUP_MAP = {
    'exact_match':        'Exact match',
    'in_frame_shorter':   'In-frame change',
    'in_frame_longer':    'In-frame change',
    'frameshift_shorter': 'Frameshift',
    'frameshift_longer':  'Frameshift',
    'coding_lost':        'Coding status change',
    'coding_gained':      'Coding status change',
    'non_coding':         'Non-coding',
}
GROUP_ORDER  = ['Exact match', 'In-frame change', 'Frameshift', 'Coding status change', 'Non-coding']
GROUP_COLORS = {
    'Exact match':        '#2ecc71',
    'In-frame change':    '#3498db',
    'Frameshift':         '#e74c3c',
    'Coding status change': '#f39c12',
    'Non-coding':         '#95a5a6',
}

ct_path = DIV_DIR / 'grch38_cds_change_type_cross_tab.tsv'
ct = pd.read_csv(ct_path, sep='\t')
pc = ct[ct['biotype'] == 'protein_coding'].copy()
if GROUPED:
    pc['ens_group'] = pc['ens_cds_change_type'].map(GROUP_MAP)
    pc['cat_group'] = pc['cat_cds_change_type'].map(GROUP_MAP)
    rows = pc.groupby(['ens_group','cat_group'])['count'].sum().reset_index()
    ens_labels = GROUP_ORDER
    cat_labels = GROUP_ORDER
    color_map = GROUP_COLORS
    title = 'CDS change type concordance (grouped, protein-coding)'
else:
    rows = pc.groupby(['ens_cds_change_type','cat_cds_change_type'])['count'].sum().reset_index()
    ens_labels = rows['ens_cds_change_type'].unique().tolist()
    cat_labels = rows['cat_cds_change_type'].unique().tolist()
    color_map = {k: '#3498db' for k in ens_labels}
    title = 'CDS change type concordance (8 types, protein-coding)'

# Pivot to a matrix
pivot = rows.pivot_table(index=rows.columns[0], columns=rows.columns[1], values='count', fill_value=0)
pivot = pivot.reindex(index=ens_labels, columns=cat_labels, fill_value=0)
total = pivot.values.sum()

# Compute cumulative ranges along y for the two vertical bars
def cumulative_ranges(totals):
    ranges, cursor = {}, 0.0
    for lab, v in totals.items():
        f = v / total if total > 0 else 0
        ranges[lab] = (cursor, cursor + f)
        cursor += f
    return ranges

ens_totals = {lab: pivot.loc[lab].sum() for lab in ens_labels}
cat_totals = {lab: pivot[lab].sum()   for lab in cat_labels}
ens_ranges = cumulative_ranges(ens_totals)
cat_ranges = cumulative_ranges(cat_totals)

fig, ax = plt.subplots(figsize=(9, 8))
X_ENS, X_CAT, BAR_W = 0.2, 0.8, 0.05

# Draw bars
for lab, (lo, hi) in ens_ranges.items():
    ax.bar(x=X_ENS, height=hi-lo, width=BAR_W, bottom=lo, color=color_map.get(lab, '#95a5a6'), align='center')
for lab, (lo, hi) in cat_ranges.items():
    ax.bar(x=X_CAT, height=hi-lo, width=BAR_W, bottom=lo, color='#bdc3c7', align='center')

# Ribbons
from matplotlib.path import Path
from matplotlib.patches import PathPatch

ens_cursor = {k: v[0] for k,v in ens_ranges.items()}
cat_cursor = {k: v[0] for k,v in cat_ranges.items()}
for e in ens_labels:
    for c in cat_labels:
        cnt = float(pivot.loc[e, c]) if (e in pivot.index and c in pivot.columns) else 0.0
        if cnt <= 0: continue
        frac = cnt / total if total > 0 else 0.0
        e_lo, e_hi = ens_cursor[e], ens_cursor[e] + frac
        c_lo, c_hi = cat_cursor[c], cat_cursor[c] + frac
        ens_cursor[e] = e_hi
        cat_cursor[c] = c_hi
        cx = (X_ENS + X_CAT) / 2
        verts = [
            (X_ENS+BAR_W/2, e_lo), (cx, e_lo), (cx, c_lo), (X_CAT-BAR_W/2, c_lo),
            (X_CAT-BAR_W/2, c_hi), (cx, c_hi), (cx, e_hi), (X_ENS+BAR_W/2, e_hi), (X_ENS+BAR_W/2, e_lo)
        ]
        codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4,
                 Path.LINETO, Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.CLOSEPOLY]
        ax.add_patch(PathPatch(Path(verts, codes), facecolor=color_map.get(e, '#95a5a6'), edgecolor='none', alpha=0.35, zorder=0))

# Labels
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xticks([X_ENS, X_CAT])
ax.set_xticklabels(['Ensembl', 'CAT'])
ax.set_yticks([])
ax.set_title(title)
plt.tight_layout()
out = OUTPUT_DIR / ('figure_main2_cds_alluvial_grouped.png' if GROUPED else 'figure_main2_cds_alluvial_ungrouped.png')
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show(); print('Saved', out)